# 04 · Consumption & Evaluation

Bring it together: answer the **core question** two ways (structured + generative),
pin a **verified query**, and run an **evaluation** — which is exactly the
"review functionality for semantic views" the team asked about.

> **The core question:** *"Compare clarity feedback across good / normal / poor
> game rounds and explain the main driver of negative sentiment."*

### Context

In [ ]:
SET sch = 'PLG_CORTEX_WORKSHOP.WS_' || REGEXP_REPLACE(CURRENT_USER(), '[^A-Za-z0-9_]', '_');
USE WAREHOUSE PLG_WORKSHOP_WH;
USE SCHEMA IDENTIFIER($sch);

### 1. The structured half — via the semantic view

In [ ]:
SELECT * FROM SEMANTIC_VIEW(
  SURVEY_ANALYSIS
  METRICS responses.avg_clarity, responses.pct_low_clarity, responses.response_count
  DIMENSIONS rounds.game_round_performance
) ORDER BY game_round_performance;

### 2. The 'why' half — a live, grounded summary over the poor group
Runs over the precomputed enriched table; no batching needed.

In [ ]:
SELECT
  AI_AGG(clarity_comment,
    'These are Dutch survey comments from players whose game round went poorly. In English, explain the single biggest driver of negative clarity sentiment, with 2 supporting examples.') AS main_driver
FROM SURVEY_ENRICHED
WHERE clarity_sentiment = 'negative';

### 3. Pin a verified query on the semantic view
Verified queries give Cortex Analyst trusted SQL for recurring questions. A
semantic view always needs its full definition, so we recreate it with an
`AI_VERIFIED_QUERIES` block on the end. **Fill the blank** with your own wording.

In [ ]:
CREATE OR REPLACE SEMANTIC VIEW SURVEY_ANALYSIS
  TABLES (
    responses AS SURVEY_RESPONSES PRIMARY KEY (response_id)
      WITH SYNONYMS ('survey','feedback','email survey'),
    rounds AS GAME_ROUNDS PRIMARY KEY (game_round_id),
    players AS PLAYER_BEHAVIOUR PRIMARY KEY (player_id)
  )
  RELATIONSHIPS (
    resp_to_round  AS responses (game_round_id) REFERENCES rounds (game_round_id),
    resp_to_player AS responses (player_id)     REFERENCES players (player_id)
  )
  FACTS (
    responses.clarity_rating AS clarity_rating,
    responses.tone_rating    AS tone_rating,
    players.replay_rate      AS replay_rate
  )
  DIMENSIONS (
    responses.brand AS brand COMMENT = 'NPL or VriendenLoterij',
    responses.email_type AS email_type SAMPLE_VALUES ('welcome','prize_notification','monthly_update','winback') IS_ENUM,
    responses.survey_date AS survey_date,
    rounds.game_round_performance AS game_round_performance
      SAMPLE_VALUES ('good','normal','poor') IS_ENUM
  )
  METRICS (
    responses.response_count  AS COUNT(responses.response_id),
    responses.avg_clarity     AS AVG(responses.clarity_rating),
    responses.avg_tone        AS AVG(responses.tone_rating),
    responses.pct_low_clarity AS AVG(IFF(responses.clarity_rating <= 2, 1, 0)) * 100
  )
  COMMENT = 'Player email-survey clarity/tone analysis (with verified query)'
  AI_SQL_GENERATION 'A poor game round means game_round_performance = ''poor''. Low clarity means clarity_rating <= 2.'
  AI_VERIFIED_QUERIES (
    clarity_by_round AS (
      QUESTION 'What is the average clarity by game round performance?'  -- >>> YOUR PART <<< try your own wording
      SQL 'SELECT game_round_performance, AVG(clarity_rating) AS avg_clarity
           FROM SURVEY_BASE GROUP BY game_round_performance ORDER BY game_round_performance'
    )
  );

_Full definition also in `reference/scaffold.sql` section 4.3._

### 4. Build a Cortex Agent (in Snowsight UI)
Notebooks build the data layer; the agent is assembled in **AI & ML → Agents →
Create agent**:

- **Tools:** add `SURVEY_ANALYSIS` (Cortex Analyst) and, if you built it,
  `SURVEY_FEEDBACK_SEARCH` (Cortex Search).
- **Orchestration instructions:** *structured / aggregate questions → Analyst;
  find-similar / emerging-theme questions → Search; open-ended "why" → summarise
  with the enriched table.*
- **Ask it the core question** and confirm it routes to Analyst for the numbers
  and explains the driver for the "why".

### 5. Evaluate — the 'semantic view review' feature
Open **AI & ML → Cortex Analyst → Evaluations**. Point an evaluation set (your
verified queries as ground truth) at `SURVEY_ANALYSIS`. It scores
**`sql_correctness`** (LLM-as-judge), tracks regressions and latency, and has an
"Improve" flow that suggests fixes.

### Checkpoint ✅
- Structured query shows clarity lowest for `poor` rounds.
- The `AI_AGG` summary names a believable main driver with examples.
- Your agent answers the core question end-to-end, and you have one eval score.

---
## Wrap — your 5 go/no-go questions (answer for yourselves)
1. **Business value** — would these answers change how you write emails?
2. **Cost** — precompute + `AI_AGG`: is the token story acceptable vs the old batching?
3. **Maintainability** — is an incremental Dynamic Table simpler to own than the map-reduce?
4. **Path to production** — governance (RBAC, lineage), per-user credit limits (see `reference/cost_guardrails.md`).
5. **Reuse** — does this pattern generalise to your other surveys / brands?

## The questions the team raised, answered
- **Good case for Cortex?** Yes.
- **Cap token usage per analyst?** Per-user monthly credit limits via `CORTEX_AI_FUNCTIONS_USAGE_HISTORY` (GA Mar 2026). Timeouts cap runtime, not tokens. See `reference/cost_guardrails.md`.
- **Semantic-view review functionality?** Cortex Analyst evaluations (`sql_correctness`) — Stage 5 above.
- **How closely must a question match a verified query?** Semantic similarity, not exact wording.
- **Are we teaching the right thing?** With precompute + aggregate functions — yes. That's the durable pattern.

Thanks for building with us. — Miriam